<a href="https://colab.research.google.com/github/AnshG-19/Blackhole_simulator_rayTracing/blob/main/notebooks/06_interactive_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys

repo_path = "/content/Blackhole_simulator_rayTracing"

if not os.path.exists(repo_path):
    %cd /content
    !git clone https://github.com/AnshG-19/Blackhole_simulator_rayTracing.git

os.chdir(repo_path)

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Repository connected:", os.getcwd())

/content
Cloning into 'Blackhole_simulator_rayTracing'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 96 (delta 34), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 2.98 MiB | 13.52 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Repository connected: /content/Blackhole_simulator_rayTracing


In [2]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import Circle
from ipywidgets import interact, FloatSlider

from src.physics.ray_tracing import calculate_light_ray

print("Interactive visualization tools loaded.")

Interactive visualization tools loaded.


In [3]:
def interactive_light_ray(b_over_M=8.0):
    M = 1.0
    b = b_over_M * M
    b_critical = 3 * np.sqrt(3) * M

    try:
        ray = calculate_light_ray(
            mass_parameter=M,
            impact_parameter=b,
            max_angle=2.6 if b_over_M < 7 else 4.0,
            points=1500,
        )
    except (ValueError, RuntimeError) as error:
        print("Could not calculate this trajectory:", error)
        return

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.plot(
        ray["x"], ray["y"],
        color="deepskyblue",
        linewidth=2,
        label=f"Light ray (b = {b_over_M:.2f} M)",
    )

    ax.add_patch(Circle((0, 0), 2 * M, color="black",
                        label="Event horizon"))
    ax.add_patch(Circle((0, 0), 3 * M, fill=False,
                        color="orange", linestyle="--",
                        linewidth=1.8, label="Photon sphere"))

    ax.set_xlim(-12, 12)
    ax.set_ylim(-12, 12)
    ax.set_aspect("equal")
    ax.set_xlabel("x / M")
    ax.set_ylabel("y / M")
    ax.set_title("Interactive Gravitational Light Bending")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

    print(f"Critical impact parameter: {b_critical:.3f} M")
    print(f"Chosen impact parameter: {b_over_M:.3f} M")
    print(f"Closest approach: {ray['closest_approach']:.3f} M")


interact(
    interactive_light_ray,
    b_over_M=FloatSlider(
        value=8.0,
        min=5.3,
        max=15.0,
        step=0.2,
        description="b / M",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=8.0, continuous_update=False, description='b / M', max=15.0, min=5.3, …

In [8]:

def compare_light_rays(max_b_over_M=12.0):
    M = 1.0
    impact_parameters = [6.0, 8.0, 10.0, float(max_b_over_M)]

    fig, ax = plt.subplots(figsize=(9, 9))

    for b_over_M in impact_parameters:
        try:
            ray = calculate_light_ray(
                mass_parameter=M,
                impact_parameter=b_over_M * M,
                max_angle=2.6,
                points=1500,
            )
        except (ValueError, RuntimeError) as error:
            print(f"Could not calculate b = {b_over_M:.1f} M: {error}")
            continue

        ax.plot(
            ray["x"],
            ray["y"],
            linewidth=2,
            label=f"b = {b_over_M:.1f} M",
        )

    ax.add_patch(
        Circle((0, 0), 2 * M, color="black", label="Event horizon")
    )
    ax.add_patch(
        Circle(
            (0, 0), 3 * M, fill=False,
            color="orange", linestyle="--", label="Photon sphere"
        )
    )

    # Keep the viewing area fixed so the trajectory's movement is comparable.
    ax.set_xlim(-25, 25)
    ax.set_ylim(-25, 25)
    ax.set_aspect("equal")
    ax.set_xlabel("x / M")
    ax.set_ylabel("y / M")
    ax.set_title("Comparing Light-Ray Trajectories")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

    print(f"Slider-selected impact parameter: {max_b_over_M:.1f} M")


interact(
    compare_light_rays,
    max_b_over_M=FloatSlider(
        value=12.0,
        min=11.0,
        max=20.0,
        step=1.0,
        description="Outer b / M",
        continuous_update=False,
    ),
);


interactive(children=(FloatSlider(value=12.0, continuous_update=False, description='Outer b / M', max=20.0, mi…

In [6]:
import ipywidgets as widgets
print("ipywidgets version:", widgets.__version__)
print("Widget support loaded.")

ipywidgets version: 7.7.1
Widget support loaded.


# Interactive Black Hole Visualization — Summary

This notebook provides interactive exploration of gravitational light bending around a Schwarzschild black hole.

### Interactive features
- Adjust the light ray's impact parameter with a slider.
- Compare trajectories with different impact parameters.
- Display the event horizon and photon sphere as geometric reference points.
- Reuse the shared numerical light-ray solver from `src/physics/ray_tracing.py`.

### Physical interpretation

The critical impact parameter is \(b_{\mathrm{crit}}=3\sqrt{3}M\) in geometric units. Escaping rays with impact parameters closer to this threshold generally experience stronger deflection.

### Limitations

The plotted paths are finite trajectory segments, not complete ray-traced images. The notebook does not yet model a luminous accretion disk, gravitational redshift, Doppler beaming, or a full relativistic camera.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import Circle
from ipywidgets import interact, FloatSlider

Define the black hole and coordinate grid
Explanation
We use geometric units so the mass scale is \(M=1\). This means the event horizon is at \(r=2\), the photon sphere at \(r=3\), and the simplified disk inner edge at \(r=6\).

X and Y represent every pixel's coordinates. R gives the distance of each pixel from the center. We'll use that distance to determine how bright each part of the disk should be.

Unlike the earlier notebooks, this grid is created once and reused whenever you move a slider.

In [ ]:
# Use geometric units: G = c = M = 1
M = 1.0

# Characteristic radii
r_s = 2 * M
r_ph = 3 * M

# Accretion disk inner edge (simplified Schwarzschild model)
r_inner = 6 * M

# Create a 2D coordinate grid
resolution = 500
view_radius = 25

x = np.linspace(-view_radius, view_radius, resolution)
y = np.linspace(-view_radius, view_radius, resolution)

X, Y = np.meshgrid(x, y)

# Distance from the black hole at every grid point
R = np.sqrt(X**2 + Y**2)

print("Event horizon radius:", r_s)
print("Photon sphere radius:", r_ph)
print("Disk inner radius:", r_inner)

Event horizon radius: 2.0
Photon sphere radius: 3.0
Disk inner radius: 6.0


Create the interactive visualization function

In [ ]:
def show_black_hole(
    outer_radius=20.0,
    brightness_scale=1.0,
    asymmetry=0.3
):
    # Only pixels inside the disk boundaries belong to the disk
    disk_mask = (R >= r_inner) & (R <= outer_radius)

    # Start with a completely dark image
    brightness = np.zeros_like(R)

    # Simplified radial brightness profile
    brightness[disk_mask] = (
        (r_inner / R[disk_mask])**3
        * (1 - np.sqrt(r_inner / R[disk_mask]))
    )

    # Add an angular pattern to make the disk less uniform
    phi = np.arctan2(Y, X)

    angular_pattern = (
        1 + asymmetry * np.cos(phi)
    )

    brightness *= angular_pattern

    # Adjust brightness without modifying the original formula
    brightness *= brightness_scale

    # Display the image
    fig, ax = plt.subplots(figsize=(8, 8))

    image = ax.imshow(
        brightness,
        extent=[
            -view_radius, view_radius,
            -view_radius, view_radius
        ],
        origin="lower",
        cmap="inferno",
        vmin=0,
        vmax=max(0.01, brightness_scale)
    )

    # Draw the event horizon
    event_horizon = Circle(
        (0, 0),
        r_s,
        color="black",
        zorder=3
    )
    ax.add_patch(event_horizon)

    # Draw the photon sphere
    photon_sphere = Circle(
        (0, 0),
        r_ph,
        fill=False,
        edgecolor="cyan",
        linestyle="--",
        linewidth=1.5,
        zorder=4
    )
    ax.add_patch(photon_sphere)

    ax.set_title(
        f"Black Hole Accretion Disk\n"
        f"Outer radius = {outer_radius:.1f} M"
    )
    ax.set_xlabel("x (units of M)")
    ax.set_ylabel("y (units of M)")
    ax.set_aspect("equal")

    fig.colorbar(image, ax=ax, label="Relative brightness")
    plt.show()

Add the sliders

In [ ]:
interact(
    show_black_hole,

    outer_radius=FloatSlider(
        value=20.0,
        min=10.0,
        max=24.0,
        step=1.0,
        description="Disk radius:",
        continuous_update=False
    ),

    brightness_scale=FloatSlider(
        value=1.0,
        min=0.2,
        max=2.0,
        step=0.1,
        description="Brightness:",
        continuous_update=False
    ),

    asymmetry=FloatSlider(
        value=0.3,
        min=0.0,
        max=0.8,
        step=0.1,
        description="Asymmetry:",
        continuous_update=False
    )
);

interactive(children=(FloatSlider(value=20.0, continuous_update=False, description='Disk radius:', max=24.0, m…

## Observations

1. Increasing the outer radius extends the disk outward.
2. Changing brightness affects the displayed intensity.
3. Increasing asymmetry makes one side brighter than the other.

## Limitations

This visualization is a simplified 2D brightness model. It does not
simulate light bending, photon capture, orbital motion, gravitational
redshift, or relativistic Doppler effects.